In [2]:
import altair as alt
import duckdb
import pandas as pd

In [2]:
alt.renderers.enable('png')

RendererRegistry.enable('png')

In [3]:
conn = duckdb.connect()

In [4]:
conn.execute("create table if not exists weather_data as select * from 'nyc_weather_data.csv'")
conn.execute("attach 'citibike_data.duckdb' as cb_data;")

In [5]:
conn.sql('describe cb_data.rides')

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ ride_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ rideable_type      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ started_at         │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ ended_at           │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ start_station_name │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ start_station_id   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ end_station_name   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ end_station_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ start_lat          │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │

In [6]:
conn.sql('describe weather_data')

┌──────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name      │ column_type │  null   │   key   │ default │  extra  │
│       varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ date                 │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ temp_max_f           │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ temp_min_f           │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ temp_mean_f          │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ feels_like_max_f     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ feels_like_min_f     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ precipitation_in     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ rain_in              │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ snowfall_in          │ DOUBLE      │ YES     │ NUL

In [7]:
df = conn.execute("""
    from cb_data.rides
    select date_trunc('month',ended_at) as month ,count(1) num_rides
    group by month
""").df()

In [8]:
alt.Chart(df).mark_bar(size=15).encode(
    alt.X('month'),
    alt.Y('num_rides'),
    tooltip=['month', 'num_rides']
).interactive()

alt.Chart(...)

In [9]:
alt.Chart(
    conn.execute("""
        with citibike_grouped_by_day as (
            select date_trunc('day',ended_at) as date,count(1) num_rides
                from cb_data.rides
            group by 1
        )
        select temp_max_f, num_rides, date
        from weather_data join citibike_grouped_by_day using(date)
        order by date asc
    """).df()
).mark_point().encode(
    x='temp_max_f',
    y='num_rides',
    tooltip=['temp_max_f', 'num_rides']
)

alt.Chart(...)

In [10]:
alt.Chart(
    conn.execute(
        """
        select extract('hour' from started_at) sa, count(1) num_rides
        from cb_data.rides
        group by sa
        order by sa asc
        """
    ).df()
).mark_line().encode(x='sa',y='num_rides').interactive()

alt.Chart(...)

In [11]:
# SELECT dayname(DATE '2026-03-11');
base = alt.Chart(
    conn.execute(
        """
        select extract('dow' FROM started_at) dow, count(1) num_rides
        from cb_data.rides
        group by dow
        order by dow asc
        """
    ).df()
).encode(x='dow:N',y=alt.Y('num_rides:Q'),tooltip=['dow','num_rides'])

base.mark_point().interactive() + base.mark_line()

#     y=alt.Y('num_rides:Q', scale=alt.Scale(domain=[4000, 8000], zero=False))

alt.LayerChart(...)

In [22]:
BOROUGHS_URL = (
    "https://raw.githubusercontent.com/codeforgermany/click_that_hood/"
    "main/public/data/new-york-city-boroughs.geojson"
)

points_df = conn.execute(
    """
    select
        start_station_name as name,
        any_value(start_lat) as lat,
        any_value(start_lng) as lon
    from 
        cb_data.rides
    where name is not null
    group by name
    """
).df()

# ── Layer 1: Borough shapes (background map) ───────────────────────────────
geo_data = alt.Data(
    url=BOROUGHS_URL,
    format=alt.DataFormat(property="features", type="json"),
)

base_map = (
    alt.Chart(geo_data)
    .mark_geoshape(fill="lightgrey", stroke="#666", strokeWidth=1.5)
)

# ── Layer 2: Point markers ─────────────────────────────────────────────────
dots = (
    alt.Chart(points_df)
    .mark_circle(size=20, opacity=0.7, color="black")
    .encode(
        longitude="lon:Q",
        latitude="lat:Q",
        tooltip=[alt.Tooltip("name:N", title="Station")],
    )
)
zoom = alt.selection_interval(bind="scales")

# ── Compose & configure ────────────────────────────────────────────────────
chart = (
    (base_map + dots)
    .add_params(zoom)
    .project(
        type="mercator",
        scale=55000,
        center=[-73.95, 40.73],
        translate=[350, 350],
    )
    .properties(
        width=700,
        height=700,
    )
    .configure_view(strokeWidth=0)
)

chart

alt.LayerChart(...)

In [23]:
import folium

m = folium.Map(location=[40.73, -73.95], zoom_start=12, tiles="CartoDB positron")

for _, row in points_df.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=3,
        color="black",
        fill=True,
        fill_opacity=0.7,
        tooltip=row["name"],
    ).add_to(m)

m